# 02 — Feature Engineering (Equities Universe)
Reads raw `.parquet` files produced by `01_fetch_training_data.ipynb` and:
1. Computes all **11 macro features** (matching `state_builder.py`).
2. Computes **5 micro features** via `compute_micro_features(..., use_proxy=True)` for training parity.
3. Computes UTC day boundaries for episode slicing.
4. Saves arrays as one compressed `.npz` per symbol for downstream notebooks/scripts.

**Input:**  `/content/drive/MyDrive/algo_trader/data/raw/{SYMBOL}.parquet`  
**Output:** `/content/drive/MyDrive/algo_trader/data/features/{SYMBOL}.npz`

In [ ]:
!pip install -q pyarrow pandas numpy pytz tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RAW_DIR  = '/content/drive/MyDrive/algo_trader/data/raw'
FEAT_DIR = '/content/drive/MyDrive/algo_trader/data/features'
os.makedirs(FEAT_DIR, exist_ok=True)
print('Directories ready')

In [ ]:
# Clone repo (or adjust path if already mounted elsewhere)
REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'
REPO_DIR = '/content/deepscalper_copilot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

COLAB_DIR = os.path.join(REPO_DIR, 'algo_trader', 'colab')
if COLAB_DIR not in sys.path:
    sys.path.insert(0, COLAB_DIR)

print('Repo path updated for deepscalper package')

In [ ]:
# Current project universe: equities pilot set (matches config.py intent)
try:
    ROOT_DIR = os.path.join(REPO_DIR, 'algo_trader')
    if ROOT_DIR not in sys.path:
        sys.path.insert(0, ROOT_DIR)
    from tickers import SP100_TICKERS
    TRADING_UNIVERSE = SP100_TICKERS[:10]
except Exception:
    TRADING_UNIVERSE = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'JPM', 'UNH', 'XOM']

print(f'Processing symbols ({len(TRADING_UNIVERSE)}): {TRADING_UNIVERSE}')

In [ ]:
import numpy as np
import pandas as pd

from deepscalper.utils import compute_macro_features, compute_micro_features, compute_day_starts

summary = []
skipped = []

for symbol in TRADING_UNIVERSE:
    raw_path = f'{RAW_DIR}/{symbol}.parquet'
    out_path = f'{FEAT_DIR}/{symbol}.npz'

    if os.path.exists(out_path):
        print(f'{symbol}: already processed - skipping.')
        skipped.append(symbol)
        continue

    if not os.path.exists(raw_path):
        print(f'WARNING: {raw_path} not found - skipping {symbol}.')
        skipped.append(symbol)
        continue

    bars = pd.read_parquet(raw_path)
    bars.columns = [c.lower() for c in bars.columns]
    bars = bars[['open', 'high', 'low', 'close', 'volume']].astype(float)
    bars = bars.sort_index()
    bars = bars[~bars.index.duplicated(keep='last')]
    bars = bars.dropna()

    macro_feats = compute_macro_features(bars)                 # (n_bars, 11)
    lob_feats   = compute_micro_features(bars, use_proxy=True) # (n_bars, 5)
    close_arr   = bars['close'].to_numpy(dtype=np.float32)
    day_starts  = np.array(compute_day_starts(bars.index), dtype=np.int32)

    if len(macro_feats) < 100:
        print(f'WARNING: {symbol} has only {len(macro_feats)} bars - skipping.')
        skipped.append(symbol)
        continue

    np.savez_compressed(
        out_path,
        macro_feats=macro_feats.astype(np.float32),
        lob_feats=lob_feats.astype(np.float32),
        close_prices=close_arr,
        day_starts=day_starts,
    )

    summary.append({
        'symbol': symbol,
        'n_bars': len(macro_feats),
        'n_days': len(day_starts),
        'out': out_path,
    })

print('\n=== FEATURE ENGINEERING SUMMARY ===')
if summary:
    print(pd.DataFrame(summary).to_string(index=False))
if skipped:
    print(f'\nSkipped symbols: {skipped}')
print('\nDone')